# 20 · Manifold learning, powered by omnibias

omnibias does not re-implement scikit-learn — it provides the *exact differential
geometry* that classical manifold-learning methods approximate. This notebook
shows two faces of that:

1. **Laplacian eigenmaps / diffusion maps** build a discrete graph Laplacian that
   approximates the continuous **Laplace–Beltrami operator** `Δ_g` — which
   omnibias computes exactly.
2. A **curvature-regularized autoencoder** uses the new **pullback metric** to
   shape the geometry of the learned latent manifold.

In [ ]:
import sys

import numpy as np
import torch
import matplotlib.pyplot as plt
from scipy.linalg import eigh

sys.path.insert(0, ".")
from _style import set_style, PRIMARY, ACCENT, GOOD
from _fields import make_field, Cos, Const
set_style()

torch.set_default_dtype(torch.float64)
rng = np.random.default_rng(0)

from omnibias.geometry import ChartSpec, ManifoldSpec
from omnibias.geometry.torch import ops as geo
from omnibias.fields.torch import _ops_dispatch as dispatch

## 1. Laplacian eigenmaps approximate `Δ_g`

We sample an S-curve 2D manifold embedded in ℝ³, build a Gaussian-affinity graph,
and embed with the eigenvectors of the normalized graph Laplacian. The result
"unrolls" the intrinsic coordinate.

In [ ]:
n = 800
t = 3 * np.pi * (rng.random(n) - 0.5)
h = 2.0 * rng.random(n)
X = np.stack([np.sin(t), h, np.sign(t) * (np.cos(t) - 1)], axis=1)

# Gaussian affinity + symmetric normalized graph Laplacian
sq = ((X[:, None, :] - X[None, :, :]) ** 2).sum(-1)
eps = np.median(sq) * 0.1
W = np.exp(-sq / eps)
np.fill_diagonal(W, 0.0)
d = W.sum(1)
Lsym = np.eye(n) - (W / np.sqrt(d[:, None] * d[None, :]))
vals, vecs = eigh(Lsym)
emb = vecs[:, 1:3]  # first two non-trivial eigenvectors

fig = plt.figure(figsize=(9.4, 3.8))
ax0 = fig.add_subplot(1, 2, 1, projection="3d")
ax0.scatter(X[:, 0], X[:, 1], X[:, 2], c=t, cmap="viridis", s=8)
ax0.set_title("S-curve in ℝ³")
ax1 = fig.add_subplot(1, 2, 2)
ax1.scatter(emb[:, 0], emb[:, 1], c=t, cmap="viridis", s=8)
ax1.set_title("Laplacian-eigenmap embedding"); ax1.set_xlabel("ψ₁"); ax1.set_ylabel("ψ₂")
plt.tight_layout()

These discrete Laplacians **converge to** the continuous `Δ_g`. omnibias gives
that operator exactly: on the unit sphere, `cos θ` is a degree-1 eigenfunction,
`Δ_g cos θ = −2 cos θ`. We verify it through a *pullback-metric* manifold.

In [ ]:
def sphere_phi(x):
    th, ph = x[0], x[1]
    return torch.stack([torch.sin(th) * torch.cos(ph),
                        torch.sin(th) * torch.sin(ph),
                        torch.cos(th)])

sphere = ManifoldSpec("S2", 2, geo.metric_spec_from_chart(
    ChartSpec(phi=sphere_phi, domain_dim=2, ambient_dim=3, name="S2")))

field = make_field(("theta", "phi"), {"f": (Cos(xp=torch), Const())}, dispatch)
th = torch.linspace(0.25, np.pi - 0.25, 80)
coords = torch.stack([th, torch.full_like(th, 0.6)], dim=-1)
lap = geo.laplace_beltrami(field(coords), "f", sphere)

fig, ax = plt.subplots(figsize=(7.0, 3.6))
ax.plot(th, lap, color=PRIMARY, label="Δ_g cos θ  (omnibias, exact)")
ax.plot(th, -2 * np.cos(th.numpy()), "--", color="k", lw=1.2, label="−2 cos θ")
ax.set_xlabel("θ"); ax.legend(); ax.set_title("The exact operator eigenmaps approximate")
print(f"max |Δ_g cosθ − (−2 cosθ)| = {float((lap + 2*torch.cos(th)).abs().max()):.2e}")
plt.tight_layout()

## 2. A curvature-regularized autoencoder (uses the pullback metric)

We fit an autoencoder to points on a curved cap. The decoder `D: ℝ² → ℝ³` is a
**chart**, so `pullback_metric(z, chart)` measures the geometry of the learned
latent manifold. Penalizing the metric distortion `‖g − I‖²` drives the chart
toward an isometric (low-curvature) embedding — and omnibias reports the exact
scalar curvature before/after.

In [ ]:
# data: a curved spherical cap in R^3
m = 256
a = rng.random(m) * 0.9
b = rng.random(m) * 2 * np.pi
cap = np.stack([np.sin(a) * np.cos(b), np.sin(a) * np.sin(b), np.cos(a)], axis=1)
data = torch.tensor(cap)

def make_ae(seed):
    torch.manual_seed(seed)
    enc = torch.nn.Sequential(torch.nn.Linear(3, 32), torch.nn.Tanh(), torch.nn.Linear(32, 2))
    dec = torch.nn.Sequential(torch.nn.Linear(2, 32), torch.nn.Tanh(), torch.nn.Linear(32, 3))
    return enc, dec

def train(lam, steps=300):
    enc, dec = make_ae(0)
    opt = torch.optim.Adam(list(enc.parameters()) + list(dec.parameters()), lr=5e-3)
    chart = ChartSpec(phi=dec, domain_dim=2, ambient_dim=3, name="dec")
    I = torch.eye(2)
    for _ in range(steps):
        opt.zero_grad()
        z = enc(data)
        recon = ((dec(z) - data) ** 2).mean()
        loss = recon
        if lam > 0:
            g = geo.pullback_metric(z, chart)
            loss = loss + lam * ((g - I) ** 2).mean()
        loss.backward(); opt.step()
    with torch.no_grad():
        z = enc(data)
        recon = float(((dec(z) - data) ** 2).mean())
    man = ManifoldSpec("ae", 2, geo.metric_spec_from_chart(chart))
    curv = float(geo.scalar_curvature(z.detach(), man).abs().mean())
    return recon, curv

r0, c0 = train(0.0)
r1, c1 = train(0.2)
print(f"plain   AE: recon={r0:.2e}  mean|R|={c0:.3f}")
print(f"reg.    AE: recon={r1:.2e}  mean|R|={c1:.3f}")

fig, ax = plt.subplots(1, 2, figsize=(8.6, 3.6))
ax[0].bar(["plain", "curv-reg"], [r0, r1], color=[PRIMARY, GOOD]); ax[0].set_title("reconstruction MSE")
ax[1].bar(["plain", "curv-reg"], [c0, c1], color=[PRIMARY, GOOD]); ax[1].set_title("mean |scalar curvature|")
plt.tight_layout()

## Takeaway

Manifold learning *sits on* omnibias: discrete Laplacians approximate the exact
`Δ_g`, and the pullback metric turns "geometry of the latent space" into a
differentiable training signal. omnibias is the foundation, not a competitor, to
these methods.